In [1]:
import os
print(os.getcwd())

/Users/apple/ml-env/footballsearch


In [3]:
import pandas as pd
import numpy as np
import json
import sys

sys.path.append('src')   # not '../src'

from evaluate import make_eval_splits, evaluate_retrieval, evaluate_by_sport, save_metrics

questions = pd.read_csv('data/processed/questions.csv')
pairs     = pd.read_csv('data/processed/pairs.csv')

print(f"Questions : {len(questions)}")
print(f"Pairs     : {len(pairs)}")

Questions : 6107
Pairs     : 944


In [6]:
# Deduplicate bidirectional pairs
# (A, B) and (B, A) are the same pair — keep only one
pairs['pair_key'] = pairs.apply(
    lambda row: tuple(sorted([row['PostId'], row['RelatedPostId']])),
    axis=1
)
pairs_deduped = pairs.drop_duplicates(subset='pair_key').drop(columns='pair_key')
pairs_deduped = pairs_deduped.reset_index(drop=True)

print(f"Pairs before dedup : {len(pairs)}")
print(f"Pairs after dedup  : {len(pairs_deduped)}")

Pairs before dedup : 944
Pairs after dedup  : 776


In [7]:
train_pairs, eval_pairs = make_eval_splits(pairs_deduped, train_ratio=0.8, random_seed=42)

train_pairs.to_csv('data/processed/train_pairs.csv', index=False)
eval_pairs.to_csv('data/processed/eval_pairs.csv',  index=False)

print(f"Train pairs : {len(train_pairs)}")
print(f"Eval pairs  : {len(eval_pairs)}")

Train pairs : 620
Eval pairs  : 156


In [8]:
train_set = set(zip(train_pairs['PostId'], train_pairs['RelatedPostId']))
eval_set  = set(zip(eval_pairs['PostId'],  eval_pairs['RelatedPostId']))

overlap = train_set & eval_set
print(f"Overlapping pairs: {len(overlap)}")
assert len(overlap) == 0, "Data leak detected — pairs appear in both splits!"

Overlapping pairs: 0


In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

baseline_model      = SentenceTransformer('all-mpnet-base-v2')
baseline_embeddings = np.load('data/processed/baseline_embeddings.npy')

class SemanticRetriever:
    def __init__(self, embeddings, questions, model):
        self.embeddings = embeddings
        self.questions  = questions
        self.model      = model

    def search(self, query, top_k=5):
        query_emb  = self.model.encode([query])
        sims       = cosine_similarity(query_emb, self.embeddings)[0]
        top_idx    = np.argsort(sims)[::-1][:top_k]
        return [
            {
                'question_id': int(self.questions.iloc[i]['Id']),
                'title'      : self.questions.iloc[i]['Title'],
                'score'      : float(sims[i]),
            }
            for i in top_idx
        ]

retriever = SemanticRetriever(baseline_embeddings, questions, baseline_model)
print("Retriever loaded.")

/Users/apple/ml-env/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Retriever loaded.


In [10]:
print(f"Embeddings shape : {baseline_embeddings.shape}")
print(f"Questions rows   : {len(questions)}")

Embeddings shape : (6107, 768)
Questions rows   : 6107


In [11]:
eval_tuples = list(eval_pairs.itertuples(index=False, name=None))

metrics = evaluate_retrieval(
    retriever    = retriever,
    eval_pairs   = eval_tuples,
    questions_df = questions,
    k_values     = [1, 5, 10],
    max_k        = 10,
)

print(json.dumps(metrics, indent=2))
save_metrics(metrics, 'results/baseline_metrics.json')

{
  "recall@1": 0.2143,
  "recall@5": 0.4156,
  "recall@10": 0.5195,
  "mrr": 0.3042,
  "total": 154
}
Saved metrics to results/baseline_metrics.json


In [12]:
breakdown = evaluate_by_sport(
    retriever    = retriever,
    eval_pairs   = eval_tuples,
    questions_df = questions,
    k            = 5,
)

# Pretty print as a table
breakdown_df = pd.DataFrame([
    {'sport': sport, **vals}
    for sport, vals in breakdown.items()
])
print(breakdown_df.to_string(index=False))

save_metrics(breakdown, 'results/baseline_sport_breakdown.json')

                                                       sport  recall@5  count
                                            |rules|football|    0.3636     11
                                |rules|football|officiating|    0.1429      7
                                                    |tennis|    0.5000      4
                                      |football|officiating|    0.2500      4
                                                   |cricket|    0.0000      4
                                             |rules|cricket|    0.0000      4
                                       |cricket|terminology|    0.0000      3
                               |rules|american-football|nfl|    0.0000      3
                                                  |football|    0.6667      3
                                                |rules|pool|    0.5000      2
                                     |american-football|nfl|    0.0000      2
                                      |rules|basketball|nba|    

In [13]:
# Extract primary sport from the first tag in the combination
def extract_primary_sport(tag_str):
    # tags look like |rules|football|officiating|
    tags = [t for t in tag_str.strip('|').split('|') if t]
    # skip meta-tags that aren't sports
    meta_tags = {'rules', 'trivia', 'history', 'officiating', 'terminology',
                 'equipment', 'training', 'technique', 'scoring', 'strategy',
                 'statistics', 'records', 'tournaments', 'media', 'injuries',
                 'technology', 'performance', 'coaching', 'schedule'}
    for tag in tags:
        if tag not in meta_tags:
            return tag
    return tags[0] if tags else 'untagged'

# Rebuild breakdown grouped by primary sport
from collections import defaultdict

sport_hits  = defaultdict(int)
sport_total = defaultdict(int)

for sport, vals in breakdown.items():
    primary = extract_primary_sport(sport)
    sport_total[primary] += vals['count']
    sport_hits[primary]  += int(round(vals['recall@5'] * vals['count']))

aggregated = {}
for sport, total in sorted(sport_total.items(), key=lambda x: -x[1]):
    aggregated[sport] = {
        'recall@5' : round(sport_hits[sport] / total, 4),
        'count'    : total,
    }

agg_df = pd.DataFrame([
    {'sport': sport, **vals}
    for sport, vals in aggregated.items()
])
print(agg_df.to_string(index=False))

save_metrics(aggregated, 'results/baseline_sport_breakdown_aggregated.json')

               sport  recall@5  count
            football    0.3922     51
             cricket    0.2609     23
   american-football    0.4118     17
              tennis    0.5833     12
            baseball    0.6000     10
          basketball    0.6667      6
            olympics    0.5000      4
        table-tennis    0.2500      4
                pool    0.3333      3
          volleyball    0.3333      3
                golf    0.0000      2
            swimming    0.5000      2
international-sports    0.5000      2
             archery    1.0000      1
  competitive-eating    1.0000      1
             running    0.0000      1
              squash    0.0000      1
             skating    0.0000      1
            training    1.0000      1
            softball    0.0000      1
             cycling    1.0000      1
           formula-1    0.0000      1
             surfing    0.0000      1
           technique    0.0000      1
                 nba    1.0000      1
            

Week 5 - Baseline comparisons

In [14]:
# Build combined text
def combine_title_body(row):
    title = str(row['Title']).strip()
    body  = str(row['Body']).strip() if pd.notna(row['Body']) else ''
    body_snippet = body[:200]
    return f"{title} {body_snippet}".strip()

combined_texts = questions.apply(combine_title_body, axis=1).tolist()

# Encode
print("Encoding title+body...")
combined_model      = SentenceTransformer('all-mpnet-base-v2')
combined_embeddings = combined_model.encode(
    combined_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Shape: {combined_embeddings.shape}")
np.save('data/processed/combined_embeddings.npy', combined_embeddings)

Encoding title+body...


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

Shape: (6107, 768)


In [15]:
combined_retriever = SemanticRetriever(combined_embeddings, questions, combined_model)

combined_metrics = evaluate_retrieval(
    retriever    = combined_retriever,
    eval_pairs   = eval_tuples,
    questions_df = questions,
    k_values     = [1, 5, 10],
    max_k        = 10,
)

print("Title only:")
print(json.dumps(metrics, indent=2))
print("\nTitle + body:")
print(json.dumps(combined_metrics, indent=2))

save_metrics(combined_metrics, 'results/combined_metrics.json')

Title only:
{
  "recall@1": 0.2143,
  "recall@5": 0.4156,
  "recall@10": 0.5195,
  "mrr": 0.3042,
  "total": 154
}

Title + body:
{
  "recall@1": 0.2792,
  "recall@5": 0.474,
  "recall@10": 0.5649,
  "mrr": 0.3633,
  "total": 154
}
Saved metrics to results/combined_metrics.json


In [16]:
# combined_metrics is now your true baseline
save_metrics(combined_metrics, 'results/baseline_metrics.json')
print("Updated baseline to title+body encoding.")

Saved metrics to results/baseline_metrics.json
Updated baseline to title+body encoding.


In [17]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    # lowercase, remove punctuation, split on whitespace
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

# Tokenize combined title+body
print("Tokenizing corpus...")
tokenized_corpus = [tokenize(text) for text in combined_texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built over {len(tokenized_corpus)} documents.")

Tokenizing corpus...
BM25 index built over 6107 documents.


In [18]:
class BM25Retriever:
    def __init__(self, bm25, questions):
        self.bm25      = bm25
        self.questions = questions

    def search(self, query, top_k=5):
        tokenized_query = tokenize(query)
        scores          = self.bm25.get_scores(tokenized_query)
        top_idx         = np.argsort(scores)[::-1][:top_k]
        return [
            {
                'question_id': int(self.questions.iloc[i]['Id']),
                'title'      : self.questions.iloc[i]['Title'],
                'score'      : float(scores[i]),
            }
            for i in top_idx
        ]

bm25_retriever = BM25Retriever(bm25, questions)
print("BM25 retriever ready.")

BM25 retriever ready.


In [19]:
bm25_metrics = evaluate_retrieval(
    retriever    = bm25_retriever,
    eval_pairs   = eval_tuples,
    questions_df = questions,
    k_values     = [1, 5, 10],
    max_k        = 10,
)

save_metrics(bm25_metrics, 'results/bm25_metrics.json')

print("Title only (dense):")
print(json.dumps(metrics, indent=2))
print("\nTitle+body (dense):")
print(json.dumps(combined_metrics, indent=2))
print("\nBM25 (sparse):")
print(json.dumps(bm25_metrics, indent=2))

Saved metrics to results/bm25_metrics.json
Title only (dense):
{
  "recall@1": 0.2143,
  "recall@5": 0.4156,
  "recall@10": 0.5195,
  "mrr": 0.3042,
  "total": 154
}

Title+body (dense):
{
  "recall@1": 0.2792,
  "recall@5": 0.474,
  "recall@10": 0.5649,
  "mrr": 0.3633,
  "total": 154
}

BM25 (sparse):
{
  "recall@1": 0.1623,
  "recall@5": 0.3766,
  "recall@10": 0.4091,
  "mrr": 0.2372,
  "total": 154
}
